# Chunking semântico multimodal dos manuais automotivos

Este notebook implementa a primeira etapa de um RAG multimodal para os PDFs da pasta `documents`: leitura dos manuais, extração de texto e imagens, criação de chunks semânticos, geração de embeddings textuais, criação de índice FAISS e salvamento dos metadados.

A estratégia não usa apenas tamanho fixo. Ela combina sinais estruturais e visuais:

- títulos e subtítulos detectados por fonte, caixa alta, numeração e padrões comuns de manual;
- continuidade temática entre páginas;
- tópicos/listas e blocos curtos de instrução;
- proximidade entre texto e imagens no layout da página;
- overlap semântico por cabeçalho e trecho final do chunk anterior quando uma seção fica longa.

## 1. Dependências

O projeto não tinha um arquivo Python de dependências na raiz, então foi criado `requirements.txt`. Execute a célula abaixo apenas se o ambiente ainda não tiver as bibliotecas instaladas.

In [1]:
# Opcional: instale as dependências no kernel atual.
# Descomente se necessário.
# %pip install -r requirements.txt

## 2. Imports e configuração

Ajuste `MAX_PDFS` para testar com poucos manuais antes de processar todos os 2.390 PDFs. Você também pode filtrar por marcas em `BRAND_FILTER`.

In [2]:
from __future__ import annotations

import hashlib
import json
import re
import statistics
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import faiss
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
DOCUMENTS_DIR = PROJECT_ROOT / "documents"
OUTPUT_DIR = PROJECT_ROOT / "rag_chunks_output"
IMAGES_DIR = OUTPUT_DIR / "images"
FAISS_INDEX_PATH = OUTPUT_DIR / "manual_chunks.faiss"
METADATA_PATH = OUTPUT_DIR / "chunks_metadata.json"
PROCESSING_REPORT_PATH = OUTPUT_DIR / "processing_report.json"

# Use None para processar todos os PDFs. Para teste inicial, use por exemplo 3 ou 10.
MAX_PDFS: int | None = 30

# Exemplo: {"chevrolet", "honda"}. Use set() para não filtrar por marca.
BRAND_FILTER: set[str] = set()

# Modelo pequeno, multilíngue e prático para português/inglês/espanhol.
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Limites flexíveis: não cortam cegamente; apenas orientam quando uma seção fica grande demais.
TARGET_CHARS = 1800
MAX_CHARS = 3200
MIN_CHARS = 350
SEMANTIC_OVERLAP_SENTENCES = 2

# Imagens muito pequenas costumam ser ícones repetidos, marcadores ou ruído de layout.
MIN_IMAGE_WIDTH = 80
MIN_IMAGE_HEIGHT = 80

# PDFs às vezes expõem a página inteira como imagem, mesmo quando o texto já foi extraído.
# Antes essas imagens eram descartadas; agora são preservadas como contexto visual de página.
FULL_PAGE_IMAGE_COVERAGE_THRESHOLD = 0.65
MIN_TEXT_BLOCKS_FOR_FULL_PAGE_CONTEXT = 3
MIN_TEXT_CHARS_FOR_FULL_PAGE_CONTEXT = 250

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pasta de documentos: {DOCUMENTS_DIR}")
print(f"Saída: {OUTPUT_DIR}")

Pasta de documentos: /home/joao/Desktop/auto-rag/documents
Saída: /home/joao/Desktop/auto-rag/rag_chunks_output


## 3. Modelos de dados

As classes abaixo mantêm informações suficientes para rastrear cada chunk até o PDF original: documento, página, posição aproximada dos blocos, imagens associadas e sinais usados na segmentação.

In [3]:
@dataclass
class TextBlock:
    document_name: str
    pdf_path: str
    page_number: int
    text: str
    bbox: tuple[float, float, float, float]
    avg_font_size: float
    max_font_size: float
    is_bold: bool
    is_heading: bool = False
    heading_score: float = 0.0


@dataclass
class ImageBlock:
    document_name: str
    pdf_path: str
    page_number: int
    image_path: str
    bbox: tuple[float, float, float, float]
    width: int
    height: int
    digest: str
    occurrence_index: int
    source: str = "embedded"
    coverage: float = 0.0


@dataclass
class SemanticChunk:
    chunk_id: str
    document_name: str
    pdf_path: str
    page_start: int
    page_end: int
    section_title: str | None
    text: str
    image_paths: list[str]
    metadata: dict[str, Any] = field(default_factory=dict)

## 4. Descoberta dos PDFs

A lista preserva a marca como metadado, pois isso costuma ser útil no RAG para filtros e avaliações.

In [4]:
def discover_pdfs(documents_dir: Path, brand_filter: set[str] | None = None, max_pdfs: int | None = None) -> list[Path]:
    """Lista PDFs, com filtro opcional por marca e limite para testes."""
    brand_filter = brand_filter or set()
    pdf_paths = []

    for pdf_path in sorted(documents_dir.glob("**/*.pdf")):
        brand = pdf_path.parent.name.lower()
        if brand_filter and brand not in brand_filter:
            continue
        pdf_paths.append(pdf_path)

    if max_pdfs is not None:
        pdf_paths = pdf_paths[:max_pdfs]

    return pdf_paths


pdf_paths = discover_pdfs(DOCUMENTS_DIR, BRAND_FILTER, MAX_PDFS)
print(f"PDFs selecionados: {len(pdf_paths)}")
for pdf_path in pdf_paths[:10]:
    print("-", pdf_path.relative_to(PROJECT_ROOT))

PDFs selecionados: 30
- documents/audi/2008-audi-a4-audi-a4_2008_bbb9f770c8cb74dba7ce.pdf
- documents/audi/2010-audi-a4-audi-a4_2010_9cbdad422581ac3d1b12.pdf
- documents/audi/2011-audi-a4-audi-a4_2011_1d6a03bb893c6fe0329d.pdf
- documents/audi/2012-audi-a4-audi-a4_2012_5381832018f825308c68.pdf
- documents/audi/2013-audi-a4-audi-a4_2013_e4cafe4bb29ecece1163.pdf
- documents/audi/2013-audi-q5-audi-q5_2013_868a91671f9140a41f39.pdf
- documents/audi/2014-audi-a4-audi-a4_2014_6a711df9f45dfe8b4316.pdf
- documents/audi/2014-audi-a6-audi-a6_2014_4155964659375821f7ca.pdf
- documents/audi/2014-audi-s6-audi-s6_2014_e7ef8755c2a8f464093c.pdf
- documents/audi/2015-audi-a3-audi-a3_2015_838ef1fb6f34ca2dc682.pdf


## 5. Extração de texto e imagens

A extração usa o layout do PyMuPDF (`page.get_text("dict")`) para capturar blocos com coordenadas. As imagens são salvas em disco com hash de conteúdo para reduzir duplicatas, e cada ocorrência preserva página e caixa de posição.

Quando um PDF não tem texto extraível, ele é registrado no relatório. Para esses casos, uma etapa futura pode integrar OCR, mas este notebook mantém o pipeline robusto sem depender de OCR externo.

In [5]:
def clean_text(text: str) -> str:
    """Normaliza espaços e hifenização sem apagar fronteiras de parágrafo."""
    text = text.replace("\u00ad", "")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def block_text_from_lines(lines: list[dict[str, Any]]) -> tuple[str, list[float], bool]:
    """Extrai texto, tamanhos de fonte e indício de negrito de um bloco do PyMuPDF."""
    parts = []
    font_sizes = []
    is_bold = False

    for line in lines:
        line_parts = []
        for span in line.get("spans", []):
            raw = span.get("text", "")
            if raw.strip():
                line_parts.append(raw)
                font_sizes.append(float(span.get("size", 0)))
                font_name = span.get("font", "").lower()
                if "bold" in font_name or "black" in font_name or "demi" in font_name:
                    is_bold = True
        if line_parts:
            parts.append(" ".join(line_parts))

    return clean_text("\n".join(parts)), font_sizes, is_bold


def estimate_page_font_stats(text_blocks: list[TextBlock]) -> dict[str, float]:
    sizes = [block.avg_font_size for block in text_blocks if block.avg_font_size > 0]
    if not sizes:
        return {"median": 10.0, "p75": 12.0}
    return {
        "median": statistics.median(sizes),
        "p75": float(np.percentile(sizes, 75)),
    }


def score_heading(text: str, avg_font_size: float, max_font_size: float, is_bold: bool, page_stats: dict[str, float]) -> float:
    """Pontua prováveis títulos usando sinais tipográficos e linguísticos."""
    if not text:
        return 0.0

    score = 0.0
    word_count = len(text.split())
    normalized = text.strip()
    letters = re.sub(r"[^A-Za-zÀ-ÿ]", "", normalized)
    uppercase_ratio = sum(1 for c in letters if c.isupper()) / max(len(letters), 1)

    if max_font_size >= page_stats["p75"] + 1.0:
        score += 2.0
    if avg_font_size >= page_stats["median"] + 1.5:
        score += 1.0
    if is_bold:
        score += 1.0
    if word_count <= 12:
        score += 1.0
    if uppercase_ratio > 0.65 and len(letters) >= 4:
        score += 1.5
    if re.match(r"^(\d+(\.\d+)*|[A-Z]\.|[IVXLC]+\.)\s+", normalized):
        score += 1.0
    if normalized.endswith(":") and word_count <= 14:
        score += 0.5
    if word_count > 22:
        score -= 2.0
    if re.search(r"\b(ATENÇÃO|CUIDADO|NOTA|AVISO|PERIGO|WARNING|CAUTION|NOTE)\b", normalized, re.I):
        score += 0.75

    return score


def safe_image_extension(ext: str | None) -> str:
    ext = (ext or "png").lower().strip(".")
    if ext in {"jpeg", "jpg", "png", "webp"}:
        return "jpg" if ext == "jpeg" else ext
    return "png"


def bbox_area(bbox: tuple[float, float, float, float]) -> float:
    """Calcula área aproximada de uma caixa do PDF."""
    width = max(0.0, bbox[2] - bbox[0])
    height = max(0.0, bbox[3] - bbox[1])
    return width * height


def classify_full_page_context_image(
    image_bbox: tuple[float, float, float, float],
    page_rect: fitz.Rect,
    page_text_blocks: list[TextBlock],
) -> tuple[bool, float, str | None]:
    """Identifica imagens de página inteira que devem ser preservadas como contexto visual."""
    page_area = max(float(page_rect.width * page_rect.height), 1.0)
    image_coverage = bbox_area(image_bbox) / page_area
    text_char_count = sum(len(block.text) for block in page_text_blocks)
    has_enough_text = (
        len(page_text_blocks) >= MIN_TEXT_BLOCKS_FOR_FULL_PAGE_CONTEXT
        and text_char_count >= MIN_TEXT_CHARS_FOR_FULL_PAGE_CONTEXT
    )

    if image_coverage >= FULL_PAGE_IMAGE_COVERAGE_THRESHOLD and has_enough_text:
        reason = (
            f"imagem cobre {image_coverage:.2%} da página e há "
            f"{len(page_text_blocks)} blocos/{text_char_count} caracteres de texto extraível"
        )
        return True, image_coverage, reason

    return False, image_coverage, None


def save_image_bytes(image_bytes: bytes, ext: str, document_stem: str, page_number: int, occurrence_index: int) -> tuple[str, str, int, int]:
    """Salva imagem com hash de conteúdo e retorna caminho relativo e dimensões."""
    digest = hashlib.sha1(image_bytes).hexdigest()
    image_ext = safe_image_extension(ext)
    image_dir = IMAGES_DIR / document_stem
    image_dir.mkdir(parents=True, exist_ok=True)
    image_path = image_dir / f"{digest[:12]}.{image_ext}"

    if not image_path.exists():
        image_path.write_bytes(image_bytes)

    try:
        with Image.open(image_path) as image:
            width, height = image.size
    except Exception:
        width, height = 0, 0

    return str(image_path.relative_to(PROJECT_ROOT)), digest, width, height


def extract_pdf_elements(pdf_path: Path) -> tuple[list[TextBlock], list[ImageBlock], dict[str, Any]]:
    """Extrai blocos de texto e imagem preservando página, coordenadas e estatísticas."""
    document_name = pdf_path.name
    document_stem = pdf_path.stem
    text_blocks: list[TextBlock] = []
    image_blocks: list[ImageBlock] = []
    report = {
        "document_name": document_name,
        "pdf_path": str(pdf_path.relative_to(PROJECT_ROOT)),
        "pages": 0,
        "text_blocks": 0,
        "image_blocks": 0,
        "warnings": [],
        "skipped_images": [],
        "full_page_context_images": [],
    }

    try:
        with fitz.open(pdf_path) as doc:
            report["pages"] = doc.page_count
            for page_index, page in enumerate(doc, start=1):
                page_dict = page.get_text("dict")
                raw_blocks = page_dict.get("blocks", [])
                page_text_blocks: list[TextBlock] = []
                image_occurrence = 0

                # Primeira passagem: coleta todo o texto da página.
                # Isso evita salvar uma página renderizada como imagem quando o texto já está disponível.
                for raw_block in raw_blocks:
                    if raw_block.get("type") != 0:
                        continue

                    bbox = tuple(float(x) for x in raw_block.get("bbox", (0, 0, 0, 0)))
                    text, font_sizes, is_bold = block_text_from_lines(raw_block.get("lines", []))
                    if not text:
                        continue
                    avg_font_size = float(statistics.mean(font_sizes)) if font_sizes else 0.0
                    max_font_size = float(max(font_sizes)) if font_sizes else 0.0
                    page_text_blocks.append(TextBlock(
                        document_name=document_name,
                        pdf_path=str(pdf_path.relative_to(PROJECT_ROOT)),
                        page_number=page_index,
                        text=text,
                        bbox=bbox,
                        avg_font_size=avg_font_size,
                        max_font_size=max_font_size,
                        is_bold=is_bold,
                    ))

                page_stats = estimate_page_font_stats(page_text_blocks)
                for block in page_text_blocks:
                    block.heading_score = score_heading(block.text, block.avg_font_size, block.max_font_size, block.is_bold, page_stats)
                    block.is_heading = block.heading_score >= 3.0
                text_blocks.extend(page_text_blocks)

                # Segunda passagem: avalia imagens com a visão completa do texto da página.
                for raw_block in raw_blocks:
                    if raw_block.get("type") != 1:
                        continue

                    bbox = tuple(float(x) for x in raw_block.get("bbox", (0, 0, 0, 0)))
                    image_bytes = raw_block.get("image")
                    if not image_bytes:
                        continue
                    image_occurrence += 1

                    is_full_page_context, image_coverage, context_reason = classify_full_page_context_image(
                        bbox, page.rect, page_text_blocks
                    )

                    rel_path, digest, width, height = save_image_bytes(
                        image_bytes=image_bytes,
                        ext=raw_block.get("ext", "png"),
                        document_stem=document_stem,
                        page_number=page_index,
                        occurrence_index=image_occurrence,
                    )
                    if width < MIN_IMAGE_WIDTH or height < MIN_IMAGE_HEIGHT:
                        report["skipped_images"].append({
                            "page": page_index,
                            "occurrence_index": image_occurrence,
                            "reason": f"imagem pequena demais ({width}x{height})",
                            "bbox": tuple(round(coord, 2) for coord in bbox),
                        })
                        continue

                    image_source = "full_page_context" if is_full_page_context else "embedded"
                    if is_full_page_context:
                        report["full_page_context_images"].append({
                            "page": page_index,
                            "occurrence_index": image_occurrence,
                            "reason": context_reason,
                            "coverage": round(image_coverage, 4),
                            "path": rel_path,
                            "size": [width, height],
                        })

                    image_blocks.append(ImageBlock(
                        document_name=document_name,
                        pdf_path=str(pdf_path.relative_to(PROJECT_ROOT)),
                        page_number=page_index,
                        image_path=rel_path,
                        bbox=bbox,
                        width=width,
                        height=height,
                        digest=digest,
                        occurrence_index=image_occurrence,
                        source=image_source,
                        coverage=image_coverage,
                    ))

    except Exception as exc:
        report["warnings"].append(f"Erro ao abrir/processar PDF: {exc}")

    if not text_blocks:
        report["warnings"].append("PDF sem texto extraível; considere OCR em etapa futura.")

    report["text_blocks"] = len(text_blocks)
    report["image_blocks"] = len(image_blocks)
    report["full_page_context_image_count"] = len(report["full_page_context_images"])
    report["skipped_image_count"] = len(report["skipped_images"])
    return text_blocks, image_blocks, report

## 6. Associação texto-imagem

A associação usa proximidade de página e posição vertical. Imagens de páginas vizinhas podem ser anexadas a chunks que atravessam páginas, o que ajuda em sequências de instruções ilustradas.

In [6]:
def bbox_center_y(bbox: tuple[float, float, float, float]) -> float:
    return (bbox[1] + bbox[3]) / 2


def associate_images_to_span(
    image_blocks: list[ImageBlock],
    page_start: int,
    page_end: int,
    text_bboxes_by_page: dict[int, list[tuple[float, float, float, float]]],
) -> list[str]:
    """Seleciona imagens semanticamente próximas ao intervalo de páginas do chunk."""
    selected: list[tuple[float, str]] = []

    for image in image_blocks:
        if image.page_number < page_start - 1 or image.page_number > page_end + 1:
            continue

        page_bboxes = text_bboxes_by_page.get(image.page_number, [])
        if not page_bboxes:
            # Se a página tem imagem mas pouco texto, preserve a relação pelo intervalo de páginas.
            distance = 0.0 if page_start <= image.page_number <= page_end else 1.0
            selected.append((distance, image.image_path))
            continue

        image_y = bbox_center_y(image.bbox)
        distances = [abs(image_y - bbox_center_y(bbox)) for bbox in page_bboxes]
        min_distance = min(distances) if distances else 0.0
        page_penalty = 0 if page_start <= image.page_number <= page_end else 500
        selected.append((min_distance + page_penalty, image.image_path))

    # Remove duplicatas mantendo ordem por proximidade.
    seen = set()
    ordered_paths = []
    for _, path in sorted(selected, key=lambda item: item[0]):
        if path not in seen:
            seen.add(path)
            ordered_paths.append(path)

    return ordered_paths

## 7. Chunking semântico

O chunking abre um novo bloco em títulos prováveis e também respeita mudanças fortes de página/seção. Quando uma seção fica longa, a divisão é feita em fronteiras naturais: parágrafos, tópicos, avisos e frases, com overlap semântico curto.

In [7]:
def split_into_sentences(text: str) -> list[str]:
    """Divide texto em frases de forma simples, suficiente para overlap semântico."""
    parts = re.split(r"(?<=[.!?])\s+(?=[A-ZÀ-Ú0-9])", text.strip())
    return [part.strip() for part in parts if part.strip()]


def semantic_overlap(text: str, sentence_count: int = SEMANTIC_OVERLAP_SENTENCES) -> str:
    sentences = split_into_sentences(text)
    return " ".join(sentences[-sentence_count:]) if sentences else ""


def is_topic_boundary(text: str) -> bool:
    """Identifica fronteiras naturais de listas, avisos e instruções."""
    return bool(re.match(r"^([•\-–]|\d+[.)]|[a-zA-Z][.)])\s+", text.strip())) or bool(
        re.match(r"^(ATENÇÃO|CUIDADO|NOTA|AVISO|PERIGO|WARNING|CAUTION|NOTE)\b", text.strip(), re.I)
    )


def build_chunk_id(document_name: str, chunk_index: int, page_start: int, page_end: int) -> str:
    base = f"{document_name}|{chunk_index}|{page_start}-{page_end}"
    digest = hashlib.sha1(base.encode("utf-8")).hexdigest()[:10]
    return f"chunk_{chunk_index:06d}_{digest}"


def finalize_chunk(
    document_name: str,
    pdf_path: str,
    chunk_index: int,
    section_title: str | None,
    blocks: list[TextBlock],
    image_blocks: list[ImageBlock],
    split_reason: str,
) -> SemanticChunk | None:
    if not blocks:
        return None

    text = clean_text("\n\n".join(block.text for block in blocks))
    if not text:
        return None

    page_start = min(block.page_number for block in blocks)
    page_end = max(block.page_number for block in blocks)
    text_bboxes_by_page: dict[int, list[tuple[float, float, float, float]]] = {}
    for block in blocks:
        text_bboxes_by_page.setdefault(block.page_number, []).append(block.bbox)

    image_paths = associate_images_to_span(image_blocks, page_start, page_end, text_bboxes_by_page)
    chunk_id = build_chunk_id(document_name, chunk_index, page_start, page_end)

    metadata = {
        "source": "pdf_manual",
        "pdf_path": pdf_path,
        "brand": Path(pdf_path).parent.name,
        "page_numbers": sorted({block.page_number for block in blocks}),
        "block_count": len(blocks),
        "char_count": len(text),
        "word_count": len(text.split()),
        "split_reason": split_reason,
        "heading_scores": [round(block.heading_score, 2) for block in blocks if block.is_heading],
        "bbox_by_page": {
            str(page): [tuple(round(coord, 2) for coord in bbox) for bbox in bboxes]
            for page, bboxes in text_bboxes_by_page.items()
        },
        "image_count": len(image_paths),
    }

    return SemanticChunk(
        chunk_id=chunk_id,
        document_name=document_name,
        pdf_path=pdf_path,
        page_start=page_start,
        page_end=page_end,
        section_title=section_title,
        text=text,
        image_paths=image_paths,
        metadata=metadata,
    )


def split_large_chunk(chunk: SemanticChunk) -> list[SemanticChunk]:
    """Divide chunks grandes em fronteiras naturais e preserva contexto via overlap."""
    if len(chunk.text) <= MAX_CHARS:
        return [chunk]

    paragraphs = [paragraph.strip() for paragraph in re.split(r"\n\n+", chunk.text) if paragraph.strip()]
    if len(paragraphs) <= 1:
        paragraphs = split_into_sentences(chunk.text)

    pieces = []
    current_parts: list[str] = []
    piece_index = 0

    for paragraph in paragraphs:
        projected = "\n\n".join(current_parts + [paragraph])
        should_split = len(projected) > TARGET_CHARS and current_parts and (is_topic_boundary(paragraph) or len(projected) > MAX_CHARS)
        if should_split:
            piece_text = clean_text("\n\n".join(current_parts))
            overlap = semantic_overlap(piece_text)
            piece_index += 1
            pieces.append((piece_index, piece_text, "large_section_semantic_split"))
            current_parts = [overlap, paragraph] if overlap else [paragraph]
        else:
            current_parts.append(paragraph)

    if current_parts:
        piece_index += 1
        pieces.append((piece_index, clean_text("\n\n".join(current_parts)), "large_section_final_piece"))

    split_chunks = []
    for piece_index, piece_text, split_reason in pieces:
        piece_digest = hashlib.sha1(f"{chunk.chunk_id}|{piece_index}".encode("utf-8")).hexdigest()[:10]
        split_chunks.append(SemanticChunk(
            chunk_id=f"{chunk.chunk_id}_part_{piece_index:02d}_{piece_digest}",
            document_name=chunk.document_name,
            pdf_path=chunk.pdf_path,
            page_start=chunk.page_start,
            page_end=chunk.page_end,
            section_title=chunk.section_title,
            text=piece_text,
            image_paths=chunk.image_paths,
            metadata={**chunk.metadata, "split_reason": split_reason, "parent_chunk_id": chunk.chunk_id, "part_index": piece_index},
        ))

    return split_chunks


def create_semantic_chunks(text_blocks: list[TextBlock], image_blocks: list[ImageBlock]) -> list[SemanticChunk]:
    """Cria chunks por títulos/seções e refina seções longas por fronteiras naturais."""
    if not text_blocks:
        return []

    text_blocks = sorted(text_blocks, key=lambda block: (block.page_number, block.bbox[1], block.bbox[0]))
    chunks: list[SemanticChunk] = []
    current_blocks: list[TextBlock] = []
    current_section_title: str | None = None
    chunk_index = 0
    document_name = text_blocks[0].document_name
    pdf_path = text_blocks[0].pdf_path

    for block in text_blocks:
        starts_new_section = block.is_heading and current_blocks and len(clean_text(" ".join(b.text for b in current_blocks))) >= MIN_CHARS
        current_text_len = len(clean_text(" ".join(b.text for b in current_blocks)))
        too_large_at_boundary = current_text_len >= MAX_CHARS and (block.is_heading or is_topic_boundary(block.text))

        if starts_new_section or too_large_at_boundary:
            chunk_index += 1
            reason = "heading_boundary" if starts_new_section else "natural_boundary_after_max_chars"
            chunk = finalize_chunk(document_name, pdf_path, chunk_index, current_section_title, current_blocks, image_blocks, reason)
            if chunk:
                chunks.extend(split_large_chunk(chunk))
            current_blocks = []

        if block.is_heading:
            current_section_title = block.text

        current_blocks.append(block)

    if current_blocks:
        chunk_index += 1
        chunk = finalize_chunk(document_name, pdf_path, chunk_index, current_section_title, current_blocks, image_blocks, "document_end")
        if chunk:
            chunks.extend(split_large_chunk(chunk))

    # Une chunks muito pequenos ao anterior quando pertencem à mesma seção, evitando fragmentos pobres para RAG.
    merged_chunks: list[SemanticChunk] = []
    for chunk in chunks:
        if merged_chunks and len(chunk.text) < MIN_CHARS and chunk.section_title == merged_chunks[-1].section_title:
            previous = merged_chunks[-1]
            previous.text = clean_text(previous.text + "\n\n" + chunk.text)
            previous.page_end = max(previous.page_end, chunk.page_end)
            previous.image_paths = list(dict.fromkeys(previous.image_paths + chunk.image_paths))
            previous.metadata["char_count"] = len(previous.text)
            previous.metadata["word_count"] = len(previous.text.split())
            previous.metadata["merged_small_chunk_ids"] = previous.metadata.get("merged_small_chunk_ids", []) + [chunk.chunk_id]
        else:
            merged_chunks.append(chunk)

    return merged_chunks

## 8. Processamento dos documentos

Esta etapa processa cada PDF selecionado, cria chunks e salva um relatório com avisos. Comece com poucos PDFs e aumente o limite depois.

In [8]:
all_chunks: list[SemanticChunk] = []
processing_reports: list[dict[str, Any]] = []

for pdf_path in tqdm(pdf_paths, desc="Processando PDFs"):
    text_blocks, image_blocks, report = extract_pdf_elements(pdf_path)
    chunks = create_semantic_chunks(text_blocks, image_blocks)
    report["chunks"] = len(chunks)
    processing_reports.append(report)
    all_chunks.extend(chunks)

print(f"Chunks gerados: {len(all_chunks)}")
print(f"PDFs com avisos: {sum(1 for report in processing_reports if report['warnings'])}")

PROCESSING_REPORT_PATH.write_text(json.dumps(processing_reports, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Relatório salvo em: {PROCESSING_REPORT_PATH.relative_to(PROJECT_ROOT)}")

# Amostra rápida para inspeção humana.
for chunk in all_chunks[:3]:
    print("\n---")
    print(chunk.chunk_id, chunk.document_name, f"p.{chunk.page_start}-{chunk.page_end}")
    print("Seção:", chunk.section_title)
    print("Imagens:", len(chunk.image_paths), chunk.image_paths[:2])
    print(chunk.text[:500])

Processando PDFs:   0%|          | 0/30 [00:00<?, ?it/s]

Chunks gerados: 25934
PDFs com avisos: 0
Relatório salvo em: rag_chunks_output/processing_report.json

---
chunk_000001_65b9a2da44 2008-audi-a4-audi-a4_2008_bbb9f770c8cb74dba7ce.pdf p.1-3
Seção: None
Imagens: 3 ['rag_chunks_output/images/2008-audi-a4-audi-a4_2008_bbb9f770c8cb74dba7ce/bd61ffd39a6b.jpg', 'rag_chunks_output/images/2008-audi-a4-audi-a4_2008_bbb9f770c8cb74dba7ce/ed03382342b5.png']
Owner's Manual

Audi A4

Audi A4 englisch 9.07

uli 2007 9:16 09

No part of this Owner's M

Date of publication: 27.06

under the laws of copyrig

Subject to alteration and

translated without the wr

This paper was bleached

For the sake of th

models. You will appreciate that we must therefore reserve the right

to alter any part of the vehicle and its equipment or technical specAUDI AG works continuously to develop and further improve all

implied by the information, illustrations or descrip

---
chunk_000002_20931b28a5 2008-audi-a4-audi-a4_2008_bbb9f770c8cb74dba7ce.pdf p.3-4
Seção: AUDI AG
Im

## 9. Embeddings textuais

O texto embutido inclui o título de seção e contexto de documento para melhorar recuperação. As imagens ficam nos metadados e são associadas pelo `chunk_id`.

In [9]:
def chunk_to_embedding_text(chunk: SemanticChunk) -> str:
    """Monta o texto usado no embedding, sem perder o conteúdo original do chunk."""
    header_parts = [
        f"Documento: {chunk.document_name}",
        f"Páginas: {chunk.page_start}-{chunk.page_end}",
    ]
    if chunk.section_title:
        header_parts.append(f"Seção: {chunk.section_title}")
    return "\n".join(header_parts) + "\n\n" + chunk.text


if not all_chunks:
    raise RuntimeError("Nenhum chunk foi gerado. Verifique os PDFs selecionados e o relatório de processamento.")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embedding_texts = [chunk_to_embedding_text(chunk) for chunk in all_chunks]

embeddings = embedding_model.encode(
    embedding_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
embeddings = embeddings.astype("float32")

print("Formato dos embeddings:", embeddings.shape)

/home/joao/Desktop/auto-rag/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/811 [00:00<?, ?it/s]

Formato dos embeddings: (25934, 384)


## 10. Índice FAISS e metadados

O índice usa similaridade por produto interno com embeddings normalizados, equivalente a busca por cosseno. A posição de cada vetor no FAISS é salva em `metadata.faiss_vector_id`, garantindo ligação correta com chunk, páginas e imagens.

In [10]:
def serialize_chunk(chunk: SemanticChunk, faiss_vector_id: int) -> dict[str, Any]:
    item = asdict(chunk)
    item["metadata"] = {
        **chunk.metadata,
        "faiss_vector_id": faiss_vector_id,
        "embedding_model": EMBEDDING_MODEL_NAME,
    }
    return item


vector_dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(vector_dimension)
index.add(embeddings)

faiss.write_index(index, str(FAISS_INDEX_PATH))
chunks_metadata = [serialize_chunk(chunk, vector_id) for vector_id, chunk in enumerate(all_chunks)]
METADATA_PATH.write_text(json.dumps(chunks_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Índice FAISS salvo em: {FAISS_INDEX_PATH.relative_to(PROJECT_ROOT)}")
print(f"Metadados salvos em: {METADATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Vetores no índice: {index.ntotal}")

Índice FAISS salvo em: rag_chunks_output/manual_chunks.faiss
Metadados salvos em: rag_chunks_output/chunks_metadata.json
Vetores no índice: 25934


## 11. Teste de busca

A célula final recarrega o FAISS e os metadados do disco, faz uma consulta de exemplo e retorna texto, metadados e imagens associadas.

In [11]:
def search_chunks(query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """Busca chunks no índice salvo e retorna metadados completos."""
    saved_index = faiss.read_index(str(FAISS_INDEX_PATH))
    saved_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    scores, vector_ids = saved_index.search(query_embedding, top_k)
    results = []
    for score, vector_id in zip(scores[0], vector_ids[0]):
        if vector_id < 0:
            continue
        item = saved_metadata[int(vector_id)]
        results.append({
            "score": float(score),
            "chunk_id": item["chunk_id"],
            "document_name": item["document_name"],
            "page_start": item["page_start"],
            "page_end": item["page_end"],
            "section_title": item.get("section_title"),
            "image_paths": item.get("image_paths", []),
            "metadata": item.get("metadata", {}),
            "text_preview": item["text"][:900],
        })
    return results


example_query = "Como trocar um pneu ou usar o macaco do veículo?"
results = search_chunks(example_query, top_k=5)

for rank, result in enumerate(results, start=1):
    print(f"\n### Resultado {rank} | score={result['score']:.4f}")
    print("chunk_id:", result["chunk_id"])
    print("documento:", result["document_name"])
    print("páginas:", f"{result['page_start']}-{result['page_end']}")
    print("seção:", result["section_title"])
    print("imagens:", result["image_paths"][:5])
    print("metadados:", {k: result["metadata"].get(k) for k in ["brand", "faiss_vector_id", "image_count", "split_reason"]})
    print("texto:", result["text_preview"].replace("\n", " "))


### Resultado 1 | score=0.6461
chunk_id: chunk_000642_027f635376
documento: 2014-audi-s6-audi-s6_2014_e7ef8755c2a8f464093c.pdf
páginas: 256-256
seção: 254
Tires and wheels
imagens: ['rag_chunks_output/images/2014-audi-s6-audi-s6_2014_e7ef8755c2a8f464093c/d14b63f7b951.jpg', 'rag_chunks_output/images/2014-audi-s6-audi-s6_2014_e7ef8755c2a8f464093c/1ee45770c7c5.jpg', 'rag_chunks_output/images/2014-audi-s6-audi-s6_2014_e7ef8755c2a8f464093c/7ebdeff50193.jpg']
metadados: {'brand': 'audi', 'faiss_vector_id': 7208, 'image_count': 3, 'split_reason': 'heading_boundary'}
texto: 254 Tires and wheels  er vehicles. This can hold true for wheels of the same vehicle type. - If the spare tire is different from the  Driving with different tires reduces vehicle handling and can lead to a loss of control. - If the spare tire is not the same as the  tires that you have mounted on your vehicle (for example winter tires or wide profile tires), then use the spare tire for a short period of time only and drive